# Feedback Descent lab

> **SealQA question:** Who holds the all-time record at the Grammys for the most wins in the album of the year category?


**Terminology.** A *candidate* is a `meta.Text` solver prompt. A pairwise evaluator chooses between two answers and emits textual feedback; an editor derives one successor prompt from that evidence. This is a small teaching implementation inspired by [Feedback Descent](https://openreview.net/forum?id=TIOFvhliLA), not a reproduction of its results or an official SealQA evaluation.

## What changes

Only the solver prompt changes. The frozen question and documents, hidden exact answer, declarations, and evaluator remain outside candidate authority. The path is two initial solves, one structured comparison, one feedback-derived edit, and one edited-prompt solve: **5 live sessions**.

## Live declaration

Bring your own user-owned SDK wrapper and identify its behavior-affecting declaration. Set `META_EVOLVE_LIVE_SDK=1` only after authenticating it and reviewing its session, tool, permission, and spending limits. Every inner run declares `INNER_BUDGET.wall_seconds = 1860`; the wrapper must honor that bound or refuse preflight. These calls may incur cost.

In [ ]:
from pathlib import Path
import os, sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
lab = root / "examples/research/sealqa_labs"
if str(lab) not in sys.path:
    sys.path.insert(0, str(lab))

from sealqa_labs import UserSdkSessionRunner
from sealqa_labs.config import *

print({
    "wrapper": WRAPPER_DECLARATION, "sessions": TOTAL_SESSIONS,
    "inner_wall_seconds": INNER_BUDGET.wall_seconds,
    "opt_in": f"{LIVE_OPT_IN}=1",
})


In [ ]:
if os.getenv(LIVE_OPT_IN) != "1":
    raise RuntimeError(
        f"Live-only lab: set {LIVE_OPT_IN}=1 after authenticating your SDK."
    )

from sealqa_labs.feedback_descent import execute

raise RuntimeError(
    "Define build_agent(request), then run(UserSdkSessionRunner(agent_factory=build_agent))."
)
print("sessions executed:", len(result.sessions))
print("usage:", result.usage)


## Visible record

The next cell exposes prompt artifacts, the preferred candidate, textual feedback, edit lineage, typed failures, exact-answer measurements, and aggregate usage. Malformed comparison output, a missing edit, timeout, or provider failure has no metric.

In [ ]:
for session in result.sessions:
    print(session.label, "failure=" + (session.failure.kind if session.failure else "none"))
print("artifacts:", result.artifacts)
print("measurements:", result.measurements)


## Live observation

No official provider capture is pending. A user may perform one authorized run with a declared wrapper. Any published capture must keep the single outcome—including a tie, regression, or typed failure—and sanitize credentials without rerunning for improvement.

## Audit and exercise

- Did the editor see only the preferred prompt and textual feedback?
- Did the evaluator-only answer enter any solver or editor request?
- Is one edited evaluation enough to claim a robust improvement?

**Change and predict:** remove the instruction to ignore distractors from candidate B. Predict the preference, feedback, and final exact score before rerunning.